# ApuCast | Experimentos de la CNN
Valida a partir del 16 de junio de 2024. Compara el starter de anomalías, la pérdida normalizada por climatología y un ensamble de tres semillas con parada temprana. Solo usa los cuatro archivos oficiales. Un directorio de salida nuevo ejecuta todo desde cero.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

In [ ]:
i = np.where(ttr == '2024050118')[0][0]   # 1 de mayo, 18 UTC (1 p. m. en Perú)
fig, ax = plt.subplots(1, 3, figsize=(13, 5))
ax[0].imshow(Xtr[i, ..., 1] - 273.15, cmap='coolwarm'); ax[0].set_title('Input GFS · T (°C)')
im = ax[1].imshow(np.flipud(Ytr[i, ..., 1]) - 273.15, cmap='coolwarm'); ax[1].set_title('Target WRF · T (°C)')
ax[2].imshow(np.flipud(dem), cmap='terrain'); ax[2].set_title('Relieve (m)')
plt.colorbar(im, ax=ax[1], shrink=0.7); plt.tight_layout(); plt.show()

In [ ]:
hour = lambda t: np.array([int(k[-2:]) for k in t])
val = ttr >= '2024061600'
trn = ~val

def climatology(Y, t):
    h = hour(t)
    return np.stack([Y[h == k].mean(0) for k in range(24)])   # (24, 70, 35, 4)

clim_trn = climatology(Ytr[trn], ttr[trn])
clim_rmse_val = np.sqrt(((clim_trn[hour(ttr[val])] - Ytr[val]) ** 2).mean(axis=(0, 1, 2)))

def local_score(pred, true):
    rmse = np.sqrt(((pred - true) ** 2).mean(axis=(0, 1, 2)))
    return 1 - (rmse / clim_rmse_val).mean(), dict(zip(VARS, rmse.round(3)))

print('Baseline 1 · climatología:', local_score(clim_trn[hour(ttr[val])], Ytr[val]))

In [ ]:
def windows(X, t):
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        prev = i
        for j, lag in enumerate([2, 1, 0]):
            idx = pos.get(k - pd.Timedelta(hours=lag), None)
            out[i, j] = X[idx] if idx is not None else X[i]
    return out   # (n, 3, 17, 13, 4)

Wtr, Wte = windows(Xtr, ttr), windows(Xte, tte)
print(Wtr.shape, Wte.shape)

In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
QUICK = os.environ.get('QUICK') == '1'     # solo para probar que todo corre
EPOCHS = 2 if QUICK else 25
torch.manual_seed(0); np.random.seed(0)

# Normalización (con estadísticas de train)
mu_x = Xtr[trn].mean((0, 1, 2)); sd_x = Xtr[trn].std((0, 1, 2))
clim_all = climatology(Ytr[trn], ttr[trn])
anom_sd = (Ytr[trn] - clim_all[hour(ttr[trn])]).std((0, 1, 2))

# Coordenadas de la grilla WRF dentro de la grilla GFS, para grid_sample
glat, glon = st['gfs_lats'], st['gfs_lons']            # GFS: fila 0 = norte
wlat, wlon = st['wrf_lats'], st['wrf_lons']            # WRF: fila 0 = sur
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32, device=dev)   # (1,70,35,2)
DEM = torch.tensor((dem - dem.mean()) / dem.std(), dtype=torch.float32, device=dev)[None, None]

class Net(nn.Module):
    def __init__(self, w=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(
            nn.Conv2d(w + 9, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, 4, 1))
    def forward(self, x, h):                     # x: (B,3,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        f = self.enc(torch.cat([x, hh], 1))
        f = F.grid_sample(f, GRID.expand(B, -1, -1, -1), align_corners=True)   # (B,w,70,35)
        f = torch.cat([f, DEM.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada

def to_t(a): return torch.tensor(a, dtype=torch.float32, device=dev)
CLIM = to_t(clim_all); ASD = to_t(anom_sd)

def prep(W, t):
    return to_t((W - mu_x) / sd_x), torch.tensor(hour(t), dtype=torch.long, device=dev)

def predict(model, W, t, bs=128):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(W), bs):
            x, h = prep(W[s:s+bs], t[s:s+bs])
            out.append((CLIM[h] + model(x, h.float()) * ASD).cpu().numpy())
    return np.concatenate(out)


In [ ]:
import json
from pathlib import Path


MAX_EPOCHS = 25
PATIENCE = 5
SEEDS = (0, 1, 2)
OUTPUT = Path(os.environ.get("APUCAST_OUTPUT", ".")).resolve()
OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINTS = OUTPUT / "checkpoints"
CHECKPOINTS.mkdir(exist_ok=True)

# Estimate the normalization on a chronological calibration block inside training.
calibration = trn & (ttr >= "2024060100")
calibration_clim = climatology(Ytr[trn & ~calibration], ttr[trn & ~calibration])
calibration_error = Ytr[calibration] - calibration_clim[hour(ttr[calibration])]
clim_mse_train = to_t(np.square(calibration_error).mean(axis=(0, 1, 2)))
del calibration_error
print("MSE de climatología para la pérdida:", dict(zip(VARS, clim_mse_train.cpu().tolist())))


def fit(seed, weighted=False, early_stop=False):
    label = f"{'weighted' if weighted else 'starter'}_{seed}"
    path = CHECKPOINTS / f"{label}.pt"
    model = Net().to(device=dev, memory_format=torch.channels_last)
    if path.exists():
        record = torch.load(path, map_location=dev, weights_only=True)
        model.load_state_dict(record["weights"])
        print(f"{label}: reutilizando época {record['epoch']}, score {record['score']:.6f}")
        return model

    torch.manual_seed(seed)
    np.random.seed(seed)
    model = Net().to(device=dev, memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    indices = np.where(trn)[0]
    schedule = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=2e-3, total_steps=MAX_EPOCHS * int(np.ceil(len(indices) / 32))
    )
    best = None
    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.time()
        model.train()
        np.random.shuffle(indices)
        for start in range(0, len(indices), 32):
            batch = indices[start:start + 32]
            x, h = prep(Wtr[batch], ttr[batch])
            anomaly = to_t(Ytr[batch]) - CLIM[h]
            predicted = model(x, h.float())
            if weighted:
                error = (predicted * ASD - anomaly).square().mean(dim=(0, 1, 2))
                loss = (error / clim_mse_train).mean()
            else:
                loss = F.mse_loss(predicted, anomaly / ASD)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            schedule.step()

        score, _ = local_score(predict(model, Wtr[val], ttr[val]), Ytr[val])
        if best is None or score > best["score"]:
            best = {
                "weights": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()},
                "score": float(score),
                "epoch": epoch,
            }
        print(f"{label} {epoch:02d}: {score:.6f} ({time.time() - started:.0f}s)", flush=True)
        if early_stop and epoch - best["epoch"] >= PATIENCE:
            print(f"{label}: early stopping en época {epoch}")
            break
    model.load_state_dict(best["weights"])
    torch.save(best, path)
    return model


starter = fit(0, early_stop=True)
weighted = fit(0, weighted=True)
ensemble = [starter, *(fit(seed, early_stop=True) for seed in SEEDS[1:])]
experiments = {
    "anomalia": [starter],
    "mse_climatologia": [weighted],
    "early_stopping_ensemble": ensemble,
}


def predict_average(models, inputs, times):
    predictions = [predict(model, inputs, times) for model in models]
    return np.mean(predictions, axis=0, dtype=np.float32)


results = {}
for name, models in experiments.items():
    score, rmse = local_score(predict_average(models, Wtr[val], ttr[val]), Ytr[val])
    results[name] = {"score": float(score), "rmse": {key: float(value) for key, value in rmse.items()}}
    print(f"{name}: {score:.6f}, delta {float(score) - 0.3377:+.6f}, RMSE {rmse}")

winner = max(results, key=lambda name: results[name]["score"])
P = predict_average(experiments[winner], Wte, tte)
sample = pd.read_csv(find("sample_submission.csv"))
positions = {stamp: index for index, stamp in enumerate(tte)}
parts = sample["id"].str.split("_", expand=True)
time_indices = parts[0].map(positions)
assert time_indices.notna().all()
rows = parts[1].astype(int)
cols = parts[2].astype(int)
submission = pd.DataFrame({"id": sample["id"]})
for channel, variable in enumerate(VARS):
    submission[variable] = P[time_indices, rows, cols, channel].round(3)
path = OUTPUT / "submission_v2.csv"
submission.to_csv(path, index=False)

saved = pd.read_csv(path)
assert len(saved) == len(sample) == 646_800
assert saved.columns.equals(sample.columns)
assert saved["id"].equals(sample["id"]) and saved["id"].is_unique
assert np.isfinite(saved[VARS].to_numpy()).all()
with (OUTPUT / "results.json").open("w", encoding="utf-8") as result_file:
    json.dump({"winner": winner, "experiments": results}, result_file, indent=2)
print(f"Ganador: {winner}. CSV verificado: {path}, {len(saved):,} filas.")
